In [1]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)

Project root: c:\Users\llam_\Bootcamp\Projecte\Proyecto-final-Data-analytics


In [2]:
clean_path = r"C:\Users\llam_\Bootcamp\Projecte\Proyecto-final-Data-analytics\data"

In [3]:
import pandas as pd

data_path = os.path.join(project_root, "data")

customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))
orders = pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv"))
payments = pd.read_csv(os.path.join(data_path, "olist_order_payments_dataset.csv"))
reviews = pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(data_path, "olist_sellers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(data_path, "olist_geolocation_dataset.csv"))
translation = pd.read_csv(os.path.join(data_path, "product_category_name_translation.csv"))

## Creamos Data set

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "translation": translation
}

## Llamamos función para overview del dataset

In [5]:
from utils.data_utils import dataset_overview, convert_columns_to_datetime

In [6]:
dataset_overview(datasets)


DATASET: customers

Shape: (99441, 5)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB

Missing values:
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

Duplicate rows: 0

Head (5 rows):
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc63

## LLamamos la función para convertir todas las variables con fechas de str a "Datetime"

In [7]:
orders_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

reviews_date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

order_items_date_cols = [
    "shipping_limit_date"
]

In [8]:
orders = convert_columns_to_datetime(orders, orders_date_cols)

reviews = convert_columns_to_datetime(reviews, reviews_date_cols)

order_items = convert_columns_to_datetime(order_items, order_items_date_cols)

In [9]:
orders.info()
reviews.info()
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------         

## Creamos nuevas variables para poder responder preguntas relacionadas con el tiempo en SQL

In [10]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] -
    orders["order_purchase_timestamp"]
).dt.days

orders["delay_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.days

orders["is_delayed"] = orders["delay_days"] > 0

orders["purchase_year"] = orders["order_purchase_timestamp"].dt.year
orders["purchase_month"] = orders["order_purchase_timestamp"].dt.month
orders["purchase_day"] = orders["order_purchase_timestamp"].dt.day
orders["purchase_hour"] = orders["order_purchase_timestamp"].dt.hour
orders["purchase_year_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

In [11]:
orders[
    [
        "order_id",
        "order_status",
        "delivery_days",
        "estimated_delivery_days",
        "delay_days",
        "is_delayed",
        "purchase_year_month"
    ]
].head()

,order_id,order_status,delivery_days,estimated_delivery_days,delay_days,is_delayed,purchase_year_month
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,8.0,15,-8.0,False,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,13.0,19,-6.0,False,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,9.0,26,-18.0,False,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,13.0,26,-13.0,False,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2.0,12,-10.0,False,2018-02


In [12]:
orders[
    ["delivery_days", "estimated_delivery_days", "delay_days"]
].describe()

,delivery_days,estimated_delivery_days,delay_days
count,96476.000000,99441.000000,96476.000000
mean,12.094086,23.403958,-11.876881
std,9.551746,8.829562,10.183854
min,0.000000,1.000000,-147.000000
25%,6.000000,18.000000,-17.000000
50%,10.000000,23.000000,-12.000000
75%,15.000000,28.000000,-7.000000
max,209.000000,155.000000,188.000000


## Miramos % de is delayed (variable para clasificar si llegó tarde o no el producto)

In [13]:
orders["is_delayed"].value_counts(normalize=True) * 100

is_delayed
False    93.428264
True      6.571736
Name: proportion, dtype: float64

## Miramos la distribución de las reviews score

In [14]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [15]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## creamos tabla especifica limpia de los pedidos enviados.

In [16]:
delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

## transformamos esta variable en dtype 0-1 para que sql la procese mejor.

In [17]:
orders["is_delayed"] = orders["is_delayed"].astype(int)

## Exportación de datos para sql

In [18]:
"""customers.to_csv(os.path.join(clean_path, "customers_clean.csv"), index=False)
orders.to_csv(os.path.join(clean_path, "orders_clean.csv"), index=False)
order_items.to_csv(os.path.join(clean_path, "order_items_clean.csv"), index=False)
payments.to_csv(os.path.join(clean_path, "payments_clean.csv"), index=False)
reviews.to_csv(os.path.join(clean_path, "reviews_clean.csv"), index=False)
products.to_csv(os.path.join(clean_path, "products_clean.csv"), index=False)
sellers.to_csv(os.path.join(clean_path, "sellers_clean.csv"), index=False)
translation.to_csv(os.path.join(clean_path, "translation_clean.csv"), index=False)

_IncompleteInputError: incomplete input (3743628217.py, line 1)